In [4]:
!git clone https://github.com/yashikadhawral/emailClassifier
%cd emailClassifier
!pip install -q -r requirements.txt

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
ART = "/content/drive/MyDrive/emailClassifier_models"
import os
os.makedirs(ART, exist_ok=True)

Cloning into 'emailClassifier'...
remote: Enumerating objects: 82, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 82 (delta 35), reused 62 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (82/82), 55.00 KiB | 18.33 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/content/emailClassifier
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 58.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 123.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 123.9 MB/s eta 0:00:00
Mounted at /content/drive


In [5]:
import glob
print(glob.glob(f"{ART}/*"))

['/content/drive/MyDrive/emailClassifier_models/word2vec_enron.model.wv.vectors.npy', '/content/drive/MyDrive/emailClassifier_models/word2vec_enron.model.syn1neg.npy', '/content/drive/MyDrive/emailClassifier_models/word2vec_enron.model', '/content/drive/MyDrive/emailClassifier_models/lstm_priority', '/content/drive/MyDrive/emailClassifier_models/priority_auto_labelled.csv', '/content/drive/MyDrive/emailClassifier_models/priority_base_labelled.csv', '/content/drive/MyDrive/emailClassifier_models/priority_labelling_subset.csv', '/content/drive/MyDrive/emailClassifier_models/distilbert_priority_checkpoints', '/content/drive/MyDrive/emailClassifier_models/distilbert_priority', '/content/drive/MyDrive/emailClassifier_models/baseline_metrics.json', '/content/drive/MyDrive/emailClassifier_models/cnn_priority', '/content/drive/MyDrive/emailClassifier_models/autoencoder', '/content/drive/MyDrive/emailClassifier_models/lstm_priority_attn']


In [ ]:
import glob
print(glob.glob(f"{ART}/lstm_priority/*"))

['/content/drive/MyDrive/emailClassifier_models/lstm_priority/vocab.pkl', '/content/drive/MyDrive/emailClassifier_models/lstm_priority/lstm_metrics.json', '/content/drive/MyDrive/emailClassifier_models/lstm_priority/model.pt']


In [ ]:
import json
print(json.load(open(f"{ART}/lstm_priority/lstm_metrics.json")))

{'model': 'lstm_priority', 'accuracy': 0.8222222222222222, 'f1_macro': 0.6255405040934757, 'params': 903791, 'inference_ms_per_example': 0.24143365926984525}


In [ ]:
import kagglehub
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
spam_path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

!mkdir -p data/processed
!python data_prep/data_prep.py --enron_csv "{path}/emails.csv" --spam_csv "{spam_path}/spam.csv" --sample_size 400 --out_dir data/processed
!python data_prep/auto_label_priority.py --csv data/processed/priority_labelling_subset.csv --out data/processed/priority_auto_labelled.csv
!python data_prep/label_priority.py --csv data/processed/priority_auto_labelled.csv --finalize

!cp data/processed/*.csv {ART}/

Using Colab cache for faster access to the 'enron-email-dataset' dataset.
Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Loading Enron dataset...
  510461 usable Enron emails parsed
Loading spam dataset...
  spam dataset: using text column 'v2', label column 'v1'
  label value counts:
v1
ham     4825
spam     747
Name: count, dtype: int64
  747 spam emails found -> auto-labelled 'Low'
Saved auto-labelled base set -> data/processed/priority_base_labelled.csv (747 rows)
Saved subset for manual labelling -> data/processed/priority_labelling_subset.csv (400 rows)

Next step: run `python auto_label_priority.py` to pre-label this subset, then `python label_priority.py` to spot-check the low-confidence ones.
Loading zero-shot classifier (device=CPU)...
config.json: 100% 1.15k/1.15k [00:00<00:00, 2.17MB/s]

model.safetensors: downloading bytes:   0% 0.00/1.63G [00:00<?, ?B/s]
model.safetensors: downloading bytes:   0% 832k/1.63G [00:00<10:54, 2.49MB/s]
model.

In [ ]:
!python training/train_priority_classifier.py \
    --base_csv data/processed/priority_base_labelled.csv \
    --manual_csv data/processed/priority_auto_labelled.csv \
    --output_dir {ART}/distilbert_priority \
    --baseline_json {ART}/baseline_metrics.json

Capping spam-derived Low rows: 747 -> 500
Final class distribution used for training:
 priority
Low       507
Medium    334
High       59
Name: count, dtype: int64
Class weights (Low/Medium/High): [0.5916473269462585, 0.8978873491287231, 5.099999904632568]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 133kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 41.1MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 57.2MB/s]
config.json: 100% 483/483 [00:00<00:00, 2.06MB/s]

model.safetensors: downloading bytes:  76% 204M/268M [00:01<00:00, 240MB/s, 16.8MB/s  ]
model.safetensors: downloading bytes:  89% 238M/268M [00:01<00:00, 212MB/s, 21.1MB/s  ]
model.safetensors: downloading bytes: 100% 250M/250M [00:02<00:00, 113MB/s, 23.1MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:02<00:00, 121MB/s, 24.8MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 3148.45it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                   

In [ ]:
!python training/train_cnn_priority.py \
    --base_csv data/processed/priority_base_labelled.csv \
    --manual_csv data/processed/priority_auto_labelled.csv \
    --word2vec_model {ART}/word2vec_enron.model \
    --output_dir {ART}/cnn_priority

device: cpu
Capping spam-derived Low rows: 747 -> 500
Final class distribution used for training:
 priority
Low       507
Medium    334
High       59
Name: count, dtype: int64
vocab size: 6400 (min_freq=2)
using existing word2vec model -> /content/drive/MyDrive/emailClassifier_models/word2vec_enron.model
embedding coverage: 6319/6400 words found in word2vec vocab
class weights (Low/Medium/High): [0.5916473269462585, 0.8978873491287231, 5.099999904632568]
model params: 731,203
epoch 0: loss=0.9915 val_acc=0.8148 val_f1_macro=0.6006 (4.0s)
epoch 1: loss=0.6825 val_acc=0.9037 val_f1_macro=0.6187 (3.2s)
epoch 2: loss=0.5251 val_acc=0.7185 val_f1_macro=0.5630 (3.1s)
epoch 3: loss=0.4278 val_acc=0.9111 val_f1_macro=0.6237 (4.2s)
epoch 4: loss=0.3245 val_acc=0.8593 val_f1_macro=0.6012 (3.8s)
epoch 5: loss=0.2476 val_acc=0.8741 val_f1_macro=0.6088 (3.1s)
epoch 6: loss=0.1874 val_acc=0.8370 val_f1_macro=0.6617 (3.2s)
epoch 7: loss=0.1422 val_acc=0.8815 val_f1_macro=0.6124 (3.8s)

--- CNN final 

In [ ]:
!python training/train_ngram_lm.py --enron_csv "{path}/emails.csv" --spam_csv "{spam_path}/spam.csv" --max_docs 30000 --out_path {ART}/ngram_lm/ngram_lm.pkl

python3: can't open file '/content/training/train_ngram_lm.py': [Errno 2] No such file or directory


In [ ]:
!python -c "import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')"
!python wsd.py

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
%cd /content/emailClassifier
!python wsd.py

/content/emailClassifier


In [ ]:
!mkdir -p data/processed
!cp {ART}/priority_base_labelled.csv {ART}/priority_auto_labelled.csv {ART}/priority_labelling_subset.csv data/processed/

In [6]:
%cd /content/emailClassifier
!git pull

/content/emailClassifier
Already up to date.


In [ ]:
!python training/train_autoencoder.py \
    --base_csv data/processed/priority_base_labelled.csv \
    --manual_csv data/processed/priority_auto_labelled.csv \
    --word2vec_model {ART}/word2vec_enron.model \
    --output_dir {ART}/autoencoder

device: cuda
vocab size: 6400 (min_freq=2)
embedding coverage: 6319/6400 words found in word2vec vocab
epoch 0: train_mse=0.038478 val_mse=0.020961
epoch 1: train_mse=0.013525 val_mse=0.009204
epoch 2: train_mse=0.009631 val_mse=0.008494
epoch 3: train_mse=0.008811 val_mse=0.007707
epoch 4: train_mse=0.008001 val_mse=0.007033
epoch 5: train_mse=0.007507 val_mse=0.006700
epoch 6: train_mse=0.007196 val_mse=0.006442
epoch 7: train_mse=0.006828 val_mse=0.006184
epoch 8: train_mse=0.006584 val_mse=0.005984
epoch 9: train_mse=0.006284 val_mse=0.005837
epoch 10: train_mse=0.006093 val_mse=0.005707
epoch 11: train_mse=0.005929 val_mse=0.005583
epoch 12: train_mse=0.005777 val_mse=0.005589
epoch 13: train_mse=0.005698 val_mse=0.005464
epoch 14: train_mse=0.005526 val_mse=0.005385
epoch 15: train_mse=0.005395 val_mse=0.005297
epoch 16: train_mse=0.005319 val_mse=0.005304
epoch 17: train_mse=0.005281 val_mse=0.005176
epoch 18: train_mse=0.005123 val_mse=0.005110
epoch 19: train_mse=0.005038 val_

In [ ]:
import kagglehub
path = kagglehub.dataset_download("wcukierski/enron-email-dataset")
spam_path = kagglehub.dataset_download("uciml/sms-spam-collection-dataset")

!python training/train_ngram_lm.py --enron_csv "{path}/emails.csv" --spam_csv "{spam_path}/spam.csv" --max_docs 30000 --out_path {ART}/ngram_lm/ngram_lm.pkl

100%|██████████| 358M/358M [00:02<00:00, 148MB/s]

Extracting files...


Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
usage: train_ngram_lm.py [-h] --enron_csv ENRON_CSV [--n N] [--k K]
                         [--sample_size SAMPLE_SIZE]
train_ngram_lm.py: error: unrecognized arguments: --spam_csv /kaggle/input/sms-spam-collection-dataset/spam.csv --max_docs 30000 --out_path /content/drive/MyDrive/emailClassifier_models/ngram_lm/ngram_lm.pkl


In [ ]:
!python -c "import nltk; nltk.download('wordnet'); nltk.download('omw-1.4')"
!python wsd.py

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


In [ ]:
!python decoding_strategies.py

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100% 511/511 [00:00<00:00, 5029.45it/s]
greedy: 
beam x4: We need to finalize the marketing spend by Friday. Please review the attached sheet and reply with comments.
top-k: Hi team, following up on the Q3 budget. We need to finalize the marketing spend by Friday. Please review the attached sheet and reply with comments.


abhi

In [ ]:
!python training/train_lstm_priority.py \
    --base_csv data/processed/priority_base_labelled.csv \
    --manual_csv data/processed/priority_auto_labelled.csv \
    --word2vec_model {ART}/word2vec_enron.model \
    --output_dir {ART}/lstm_priority \
    --baseline_json {ART}/baseline_metrics.json

device: cuda
Capping spam-derived Low rows: 747 -> 500
Final class distribution used for training:
 priority
Low       507
Medium    334
High       59
Name: count, dtype: int64
vocab size: 6400 (min_freq=2)
using existing word2vec model -> /content/drive/MyDrive/emailClassifier_models/word2vec_enron.model
embedding coverage: 6319/6400 words found in word2vec vocab
class weights (Low/Medium/High): [0.5916473269462585, 0.8978873491287231, 5.099999904632568]
model params: 876,291
epoch 0: loss=1.0456 val_acc=0.9037 val_f1_macro=0.6193 (1.7s)
epoch 1: loss=0.6663 val_acc=0.5778 val_f1_macro=0.3983 (0.5s)
epoch 2: loss=0.6029 val_acc=0.7333 val_f1_macro=0.5836 (0.5s)
epoch 3: loss=0.5542 val_acc=0.9037 val_f1_macro=0.6730 (0.5s)
epoch 4: loss=0.4762 val_acc=0.6889 val_f1_macro=0.5400 (0.5s)
epoch 5: loss=0.4092 val_acc=0.8074 val_f1_macro=0.5676 (0.5s)
epoch 6: loss=0.2951 val_acc=0.8074 val_f1_macro=0.5693 (0.5s)
epoch 7: loss=0.1808 val_acc=0.8222 val_f1_macro=0.6067 (0.5s)

--- LSTM fina

In [ ]:
!python training/train_lstm_priority.py \
    --base_csv data/processed/priority_base_labelled.csv \
    --manual_csv data/processed/priority_auto_labelled.csv \
    --word2vec_model {ART}/word2vec_enron.model \
    --use_attention \
    --output_dir {ART}/lstm_priority_attn \
    --baseline_json {ART}/baseline_metrics.json

device: cuda
Capping spam-derived Low rows: 747 -> 500
Final class distribution used for training:
 priority
Low       507
Medium    334
High       59
Name: count, dtype: int64
vocab size: 6400 (min_freq=2)
using existing word2vec model -> /content/drive/MyDrive/emailClassifier_models/word2vec_enron.model
embedding coverage: 6319/6400 words found in word2vec vocab
class weights (Low/Medium/High): [0.5916473269462585, 0.8978873491287231, 5.099999904632568]
model params: 876,548
epoch 0: loss=1.0389 val_acc=0.8444 val_f1_macro=0.5802 (1.1s)
epoch 1: loss=0.7031 val_acc=0.8741 val_f1_macro=0.5997 (0.6s)
epoch 2: loss=0.6283 val_acc=0.6370 val_f1_macro=0.4673 (0.6s)
epoch 3: loss=0.5591 val_acc=0.8074 val_f1_macro=0.5953 (0.6s)
epoch 4: loss=0.5023 val_acc=0.9037 val_f1_macro=0.6225 (0.8s)
epoch 5: loss=0.4959 val_acc=0.8741 val_f1_macro=0.6083 (0.8s)
epoch 6: loss=0.4593 val_acc=0.8370 val_f1_macro=0.6177 (0.8s)
epoch 7: loss=0.4597 val_acc=0.8815 val_f1_macro=0.6106 (0.8s)

--- LSTM fina

In [10]:
!python decoding_strategies.py

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 
Loading weights: 100% 511/511 [00:00<00:00, 4970.19it/s]
greedy: 
beam x4: We need to finalize the marketing spend by Friday. Please review the attached sheet and reply with comments.
top-k: Hi team, following up on the Q3 budget. We need to finalize the marketing spend by Friday. Please review the attached sheet and reply with comments.


In [11]:
!grep -n "forced_bos\|generation_config" decoding_strategies.py

29:    forced_bos = getattr(model.config, "forced_bos_token_id", None)
33:        if step == 0 and forced_bos is not None:
34:            nxt = forced_bos  # matches generate()'s ForcedBOSTokenLogitsProcessor


In [9]:
%cd /content/emailClassifier
from wsd import demo_on_emails
from nltk import word_tokenize
import nltk
nltk.download('punkt_tab', quiet=True)

sample_emails = [
    "please confirm the interest rate on the loan before we close the account",
    "the bank will charge a settlement fee if we close this account early",
    "what is your interest in this project and can we settle on a rate",
]
tokenized = [word_tokenize(s) for s in sample_emails]

for r in demo_on_emails(tokenized):
    print(r['word'], '->', r['synset'], '-', r['definition'])

/content/emailClassifier
interest -> interest.n.04 - a fixed charge for borrowing money; usually a percentage of the amount borrowed
interest -> pastime.n.01 - a diversion that occupies one's time and thoughts (usually pleasantly)
charge -> charge.v.24 - energize a battery by passing a current through it in the direction opposite to discharge
close -> conclude.v.04 - come to a close
close -> conclude.v.04 - come to a close
rate -> pace.n.03 - the relative speed of progress or change
rate -> rate.n.04 - a quantity or amount or measure considered as a proportion of another quantity or amount or measure
bank -> savings_bank.n.02 - a container (usually with a slot in the top) for keeping money at home
account -> report.n.03 - a short account of the news
account -> report.n.03 - a short account of the news
settlement -> colony.n.01 - a body of people who settle far from home but maintain ties with their homeland; inhabitants remain nationals of their home state but are not literally under t

In [15]:
from google.colab import files
for f in ["baseline_metrics.json", "lstm_priority/lstm_metrics.json", "lstm_priority_attn/lstm_metrics.json",
          "cnn_priority/cnn_metrics.json", "autoencoder/reconstruction_metrics.json"]:
    files.download(f"{ART}/{f}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
!mkdir -p results
!cp {ART}/baseline_metrics.json {ART}/lstm_priority/lstm_metrics.json {ART}/lstm_priority_attn/lstm_metrics.json {ART}/cnn_priority/cnn_metrics.json {ART}/autoencoder/reconstruction_metrics.json results/
!git add results/
!git commit -m "Add training results: DistilBERT, LSTM, LSTM+attention, CNN, autoencoder"
!git push

cp: will not overwrite just-created 'results/lstm_metrics.json' with '/content/drive/MyDrive/emailClassifier_models/lstm_priority_attn/lstm_metrics.json'
Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@2e0750647d82.(none)')
fatal: could not read Username for 'https://github.com': No such device or address
